# Hello World: Quantum Computing with Perceval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Quandela/Tutorials/blob/main/Photonics_with_Perceval/HelloWorldPerceval.ipynb)

In this tutorial, you will build a small photonic circuit, simulate it locally, and run the same experiment on Quandela's quantum hardware. Run the cells from top to bottom; the only value you must provide is your Quandela Cloud token.

You will prepare two photons, make them interfere at a beam splitter, submit the circuit to a QPU, and interpret the measured output probability distribution.

## 1. Install and import the dependencies

The next cells prepare Google Colab or a local Jupyter environment. If prompted after a first installation, restart the kernel and continue from the top.

In [ ]:
# Supporting code: installs Perceval in the active notebook environment. You can just run this cell.
!pip install perceval-quandela

### Check the Perceval version

Let's check which version of Perceval is installed to ensure compatibility.

In [ ]:
# Supporting code: imports packages and applies the shared visual style. You can just run this cell.
import matplotlib.pyplot as plt
import perceval as pcvl
from perceval.algorithm import Sampler
from perceval import RemoteProcessor
from tqdm.notebook import tqdm
import time
from pprint import pprint

PRIMARY = "#435BEC"     # Indigo: inputs, transformations, and graph bars
ACCENT = "#F07362"      # Coral: selected values and key outcomes
BACKGROUND = "#000000"  # Black: figure backgrounds
FOREGROUND = "#FFFFFF"  # White: labels and outlines

plt.rcParams.update({
    "figure.facecolor": BACKGROUND,
    "axes.facecolor": BACKGROUND,
    "axes.edgecolor": FOREGROUND,
    "axes.labelcolor": FOREGROUND,
    "text.color": FOREGROUND,
    "xtick.color": FOREGROUND,
    "ytick.color": FOREGROUND,
})

def plot_distribution(distribution, title):
    """Display a state distribution with probabilities summing to one."""
    labels = [str(state) for state in distribution]
    values = list(distribution.values())
    total = sum(values)
    probabilities = [value / total for value in values]
    colours = [PRIMARY] * len(values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, probabilities, color=colours)
    ax.set(title=title, xlabel="Measured photon state", ylabel="Probability")
    ax.set_ylim(0, 1)
    plt.show()

print(f"Perceval version: {pcvl.__version__}")

## 2. Build and simulate a simple quantum circuit

We prepare one photon in each of two optical modes, written $|1,1\rangle$, and send both photons into a balanced beam splitter. Because the photons are indistinguishable, their probability amplitudes interfere: ideally, they leave together as $|2,0\rangle$ or $|0,2\rangle$, rather than separately as $|1,1\rangle$. This is the Hong–Ou–Mandel effect.

We first simulate the experiment locally so the circuit and expected output distribution are clear before sending it to the QPU.

In [ ]:
# Prepare one photon in each input mode: |1,1⟩.
input_state = pcvl.BasicState('|1,1>')

# Build a two-mode circuit containing one balanced beam splitter.
circuit = pcvl.Circuit(2)
circuit.add(0, pcvl.BS())

# Repeat the measurement to estimate the output probabilities.
N_SAMPLES = 10_000

# Optional: uncomment these two lines later to explore a noisy simulation.
# noise_model = pcvl.NoiseModel(brightness=1, indistinguishability=1, g2=0)

# Combine the circuit and input state into a Perceval experiment.
experiment = pcvl.Experiment(m_circuit=circuit)
experiment.with_input(input_state)
experiment.min_detected_photons_filter(1)
# experiment.noise = noise_model

# Run the circuit locally with Perceval's exact SLOS simulator.
processor = pcvl.Processor("SLOS", experiment)

# Ask for both sampled counts and exact probabilities.
sampler = Sampler(processor)
samples = sampler.sample_count(N_SAMPLES)['results']
probs = sampler.probs()['results']  # Ask for the exact probabilities

# The sampled and exact distributions should be close for 10,000 repetitions.
print("Samples:")
pcvl.pdisplay(samples)
print("Exact probabilities:")
pcvl.pdisplay(probs)
plot_distribution(probs, "Local simulation: exact output probabilities")

## 3. Configure cloud access

To run this experiment on real quantum hardware, you need an API token from Quandela Cloud.

1. Sign in to [Quandela Cloud](https://cloud.quandela.com), or create an account.
2. Open the Cloud Dashboard and create a token.

![Token page in Quandela Cloud](https://raw.githubusercontent.com/Quandela/Tutorials/main/img/CloudToken.png)

3. Paste the token into the next cell, keeping the quotation marks.

> Keep your token private: do not save it in a public notebook or commit it to Git.

In [ ]:
# ACCESS — this is the only required change.
CLOUD_TOKEN = "YOUR_CLOUD_TOKEN"  # Replace with your Quandela Cloud token

# QPU — Belenos runs this two-photon, two-mode experiment.
CLOUD_PLATFORM = "qpu:ascella"  # You can also use "qpu:ascella".

print(f"Ready to connect to {CLOUD_PLATFORM}.")

## 4. Run the experiment on a quantum computer

The next cell sends the same circuit and input state to the selected QPU. The hardware repeats the experiment `N_SAMPLES` times; on every run, photon detectors record one output state. Those repeated measurements form the probability distribution we inspect below.

In [ ]:
# Connect to the selected QPU using your token.
remote_processor = RemoteProcessor(CLOUD_PLATFORM, token=CLOUD_TOKEN).add(0, experiment)

# Send the same |1,1⟩ input used in the local simulation.
remote_processor.with_input(input_state)
remote_processor.min_detected_photons_filter(1)

# Submit the measurements asynchronously so the notebook remains responsive.
remote_sampler = Sampler(remote_processor, max_shots_per_call=1_000_000)
remote_sampler.default_job_name = "Hello World"
remote_job = remote_sampler.sample_count.execute_async(N_SAMPLES)
print("Job submitted to Quandela Cloud.")
print("Job ID:", remote_job.id)

## 5. Monitor job progress

The job appears in the **Jobs** tab of Quandela Cloud. It may wait briefly while the QPU becomes available.

![Cloud job status](https://raw.githubusercontent.com/Quandela/Tutorials/main/img/JobStatus.png)

In [ ]:
# Supporting code: checks the current cloud job status. You can just run this cell.
print(f"Job status = {remote_job.status()}")

#### Possible job statuses

- **Waiting** — the job is queued for processing.
- **Running** — the QPU is performing the measurements.
- **Completed** — the measurements finished successfully.

In [ ]:
# Supporting code: waits for the QPU job and displays its progress. You can just run this cell.
previous_prog = 0
with tqdm(total=1, bar_format='{desc}{percentage:3.0f}%|{bar}|') as tq:
    tq.set_description(f'Get {N_SAMPLES} samples from {remote_processor.name}')
    while not remote_job.is_complete:
        tq.update(remote_job.status.progress / 100 - previous_prog)
        previous_prog = remote_job.status.progress / 100
        time.sleep(1)
    tq.update(1 - previous_prog)
    tq.close()

print(f"Job status = {remote_job.status()}")

## 6. Retrieve and understand the results

The QPU returns a count for every measured photon state. Dividing each count by the total number of retained measurements turns the counts into a probability distribution. For an ideal balanced beam splitter, most probability should appear at $|2,0\rangle$ and $|0,2\rangle$; real hardware can show small deviations because of photon loss and experimental imperfections.

In [ ]:
# Download the measured counts from the completed QPU job.
results = remote_job.get_results()
remote_counts = results['results']

# Show both the Perceval table and a normalized probability distribution.
pcvl.pdisplay(remote_counts)
plot_distribution(remote_counts, f"QPU measurement distribution on {CLOUD_PLATFORM}")

### Optional: retrieve a job after reopening the notebook

If you closed the notebook while the job was queued, rerun the installation, import, and cloud-settings cells above. Then copy the job ID from Quandela Cloud into the cell below.

In [ ]:
# Supporting code: retrieves a previous cloud job. Run this cell only if you reopened the notebook.
JOB_ID = "YOUR_JOB_ID"  # Paste the job ID shown in Quandela Cloud.
resumed_processor = RemoteProcessor(CLOUD_PLATFORM, token=CLOUD_TOKEN)
resumed_job = resumed_processor.resume_job(JOB_ID)

resumed_counts = resumed_job.get_results()['results']
pcvl.pdisplay(resumed_counts)
plot_distribution(resumed_counts, f"Retrieved QPU distribution on {CLOUD_PLATFORM}")

## 7. Explore the circuit and QPU

The following supporting cells reveal more about the circuit and the selected QPU. You can run them without changing anything.

In [ ]:
# Supporting code: draws the optical circuit used in this experiment. You can just run this cell.
pcvl.pdisplay(circuit)

You can also inspect current performance information for the selected QPU:

- **Clock** is the photon-source repetition rate; 5 MHz means five million cycles per second.
- **Transmittance** estimates how often a generated photon reaches a detector instead of being lost.
- **Status** indicates whether the QPU is available, running, in maintenance, or being calibrated.

In [ ]:
# Supporting code: shows live performance information for the selected QPU. You can just run this cell.
performance = remote_processor.performance
pprint(f"The clock speed of {remote_processor.name} is {performance['Clock (MHz)']} MHz.")
pprint(f"The transmittance of {remote_processor.name} is {performance['Transmittance (%)']}%.")
pprint(f"The status of {remote_processor.name} is {remote_processor.status}.")

The architecture view shows how the optical components are arranged on the selected photonic processor.

In [ ]:
# Supporting code: displays the selected processor's optical architecture. You can just run this cell.
specs = remote_processor.specs
pcvl.pdisplay(specs.architecture, recursive=True)


The platform constraints show the circuit sizes and photon counts accepted by the selected QPU.

In [ ]:
# Supporting code: lists the selected QPU's input constraints. You can just run this cell.
print("Platform constraints:")
pprint(specs.constraints)

# What you achieved

You have now run the same photonic experiment in two places: first in a classical simulator, then on a real QPU. Perceval described the input state and optical circuit, Quandela Cloud scheduled the hardware run, and photon detectors produced the measured output distribution.

The key quantum event was two-photon interference at the beam splitter. Compare the local and QPU distributions: their overall shapes should agree, while small differences show that real experiments are physical measurements rather than perfect mathematical models.